# **CREATING ALL INPUT FILES TO RUN SCENICPLUS**

## Import tools

In [ ]:
import scenicplus
scenicplus.__version__

In [ ]:
import pycisTopic 
pycisTopic.__version__

In [ ]:
import scanpy as sc
import pandas as pd

In [ ]:
pd.set_option('display.max_columns', None) # show max n of columns when printing an obj

## Create Directories

In [ ]:
# project directory
import os

# set working directory 
work_dir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18"

# output directory
outDir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/output"

# project directory
projDir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/"


# set temporary directory
tmpDir = "/mnt/scratch/maithe_2025/SCENICplus_withoutC18_temp/"

# **PREPARE RNA-SEQ DATA**

## Import RNA data

In [ ]:
# Import RNA data 
adata_diet = sc.read_h5ad(work_dir+"/input/forSCENIC_Epi_without_C18_Diet_seurat-wnn-integrated-SCTAssay_counts.h5ad")
adata_diet.var_names_make_unique()
adata_diet
adata_diet.obs_names # check barcodes 

In [ ]:
# Split index to create "Barcode" column
adata_diet.obs["barcode"] = [x.rsplit("_", 1)[1] for x in adata_diet.obs.index]
adata_diet.obs["barcode"]

In [ ]:
# Split index to create "Sample" column
adata_diet.obs["Sample"] = [x.rsplit("_", 1)[0] for x in adata_diet.obs.index]
adata_diet.obs

## Normalization

In [ ]:
# save the non-normalized and non-scaled matrix in the raw slot 
adata_diet.raw = adata_diet

In [ ]:
sc.pp.normalize_total(adata_diet, target_sum=1e4)
sc.pp.log1p(adata_diet)
sc.pp.highly_variable_genes(adata_diet, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata_diet = adata_diet[:, adata_diet.var.highly_variable]
sc.pp.scale(adata_diet, max_value=10)

In [ ]:
print(adata_diet.obs)

In [ ]:
sc.tl.pca(adata_diet)
sc.pl.pca(adata_diet, color = "Annotated")

In [ ]:
sc.pp.neighbors(adata_diet)
sc.tl.umap(adata_diet)
sc.pl.umap(adata_diet, color = "Annotated")

In [ ]:
adata_diet.write(work_dir + "/adata_diet.h5ad")

# **PREPARE ATAC-SEQ DATA**  
Partially following this tutorial: https://pycistopic.readthedocs.io/en/latest/notebooks/human_cerebellum.html

## Create cisTopic object 

In [ ]:
# Adding cell information to cistopic obj
cell_data =  pd.read_csv(work_dir+'/input/forSCENIC_Epi_without_C18_Diet_seurat-wnn-integrated.tsv', sep='\t')
cell_data

## Get chromosome sizes (for mm10 here)

In [ ]:
import pyranges as pr
import requests
target_url='http://hgdownload.cse.ucsc.edu/goldenPath/mm10/bigZips/mm10.chrom.sizes'
chromsizes=pd.read_csv(target_url, sep='\t', header=None)
chromsizes.columns=['Chromosome', 'End']
chromsizes['Start']=[0]*chromsizes.shape[0]
chromsizes=chromsizes.loc[:,['Chromosome', 'Start', 'End']]

chromsizes

## Call peaks

In [ ]:
# create folders in the output directory
os.mkdir(outDir + '/consensus_peak_calling')
os.mkdir(outDir + '/consensus_peak_calling/pseudobulk_bed_files/')
os.mkdir(outDir + '/consensus_peak_calling/pseudobulk_bw_files/')

In [ ]:
# locate fragment files
B1_frag_path = "/mnt/scratch/maithe/Gastric_Origins_Multiome/Project_GCG_15764_B01_SCR_Lane/Sample_GastricB1-GEX/compbio/cellranger_override/outs/atac_fragments.tsv.gz"
P1_frag_path = "/mnt/scratch/maithe/Gastric_Origins_Multiome/Project_GCG_15614_B01_SCR_Lane/Sample_Ms-Stom-Pool-Gastric-Grant-Pilot-1-GEX/compbio/cellranger_override/outs/atac_fragments.tsv.gz"

In [ ]:
# generate the pseudobulk profiles
  # pseudobulk_bed_files: pseudobulk profiles stored in bed format.
  # pseudobulk_bw_files: pseudobulk profiles stored in BigWig format. Useful for visualization in IGV or UCSC genome browser

# create a dictionary of fragment files 
adata_diet.obs["Sample"] #check sample's names

In [ ]:
fragments_dict = {
    "Batch1": B1_frag_path,
    "Pilot1" : P1_frag_path
    }
fragments_dict

In [ ]:
fragments_dict

In [ ]:
# set everything up to call peaks based on cell type (feed in information from cell_data annotation)
from pycisTopic.pseudobulk_peak_calling import export_pseudobulk

bw_paths, bed_paths = export_pseudobulk(
                 input_data = cell_data,
                 variable = 'Cellannotation',   # variable by which to generate pseubulk profiles, in this case we want pseudobulks per celltype
                 sample_id_col = 'Sample',
                 chromsizes = chromsizes,
                 bed_path = os.path.join(outDir, 'consensus_peak_calling/pseudobulk_bed_files/'),  # specify where pseudobulk_bed_files should be stored
                 bigwig_path = os.path.join(outDir, 'consensus_peak_calling/pseudobulk_bw_files/'),# specify where pseudobulk_bw_files should be stored
                 path_to_fragments = fragments_dict,                                                        # location of fragment fiels                                                                                 # specify the number of cores to use, we use ray for multi processing
                 normalize_bigwig = True,
                 temp_dir = tmpDir,
                 split_pattern = '-')

In [ ]:
# Peak calling using MACS3
from pycisTopic.pseudobulk_peak_calling import peak_calling

macs_path = "/home/maithe_2025/apps/miniconda3/envs/macs3/bin/macs3"
macs_outdir = outDir + '/consensus_peak_calling/MACS/'
os.mkdir(macs_outdir)
# Run peak calling
narrow_peaks_dict = peak_calling(macs_path,
                                 bed_paths,
                                 macs_outdir,
                                 genome_size='mm',
                                 input_format='BEDPE',
                                 shift=73, 
                                 ext_size=146,
                                 keep_dup = 'all',
                                 q_value = 0.05,
                                 _temp_dir = tmpDir + 'ray_spill')

## Get consensus peaks

Downloaded blacklist file from here https://github.com/aertslab/pycisTopic/tree/old/blacklist and transfer it to current input folder

In [ ]:
from pycisTopic.iterative_peak_calling import get_consensus_peaks

In [ ]:
# params
peak_half_width=250
path_to_blacklist="/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus/input/mm10-blacklist.v2.bed"

In [ ]:
consensus_peaks = get_consensus_peaks(
    narrow_peaks_dict = narrow_peaks_dict,
    peak_half_width = peak_half_width,
    chromsizes = chromsizes,
    path_to_blacklist = path_to_blacklist)

consensus_peaks.to_bed(
    path = os.path.join(outDir, "consensus_peak_calling/consensus_regions.bed"),
    keep =True,
    compression = 'infer',
    chain = False)

## QC ATAC dataset

In [ ]:
QC_DIR = os.path.join(outDir, "QC")
os.mkdir(QC_DIR)

In [ ]:
# get TSS annotation from Ensembl Biomart
# check the latest annotation for mouse, which is mm39
# but since we used mm10 for cellranger, we need to use mm10 here too

!pycistopic tss gene_annotation_list | grep -i Mouse

In [ ]:
# checking mm10
!pycistopic tss gene_annotation_list --filter mouse --server http://nov2020.archive.ensembl.org/

In [ ]:
QC_DIR

In [ ]:
!pycistopic tss get_tss --name mmusculus_gene_ensembl --server http://nov2020.archive.ensembl.org/ --ucsc mm10 --to-chrom-source ucsc --output /mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/output/QC/mm10tss.bed

In [ ]:
!head /mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/output/QC/mm10tss.bed | column -t

In [ ]:
# get fragments path again to run pycistopic qc below
fragments_dict

In [ ]:
# Batch 1
!pycistopic qc --fragments /mnt/scratch/maithe/Gastric_Origins_Multiome/Project_GCG_15764_B01_SCR_Lane/Sample_GastricB1-GEX/compbio/cellranger_override/outs/atac_fragments.tsv.gz --regions /mnt/data/maithe/Gastric-Origins-Project/Gastric_Origin_Multiome/Healthy_Analyses-P1_B1/SCENICplus_Healthy/output/consensus_peak_calling/consensus_regions.bed --tss /mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/output/QC/mm10tss.bed --output /mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/output/QC/Batch1
    
# Pilot 1
!pycistopic qc --fragments /mnt/scratch/maithe/Gastric_Origins_Multiome/Project_GCG_15614_B01_SCR_Lane/Sample_Ms-Stom-Pool-Gastric-Grant-Pilot-1-GEX/compbio/cellranger_override/outs/atac_fragments.tsv.gz  --regions /mnt/data/maithe/Gastric-Origins-Project/Gastric_Origin_Multiome/Healthy_Analyses-P1_B1/SCENICplus_Healthy/output/consensus_peak_calling/consensus_regions.bed --tss /mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/output/QC/mm10tss.bed --output /mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/output/QC/Pilot1


In [ ]:
# Plot QC metrics
from pycisTopic.plotting.qc_plot import plot_sample_stats, plot_barcode_stats
import matplotlib.pyplot as plt
 
 # Function to generate, display, and save the plot
def generate_and_save_plots(fragments_dict, QC_DIR):
    for sample_id in fragments_dict:
        fig = plot_sample_stats(
            sample_id=sample_id,
            pycistopic_qc_output_dir=QC_DIR
        )
        fig_path = os.path.join(QC_DIR, f'{sample_id}_qc_plot.png')
        fig.savefig(fig_path)  # Save the plot as a file
        print(f'Plot saved to {fig_path}')
        plt.show()  # Display the plot

generate_and_save_plots(fragments_dict, QC_DIR)

In [ ]:
from pycisTopic.qc import get_barcodes_passing_qc_for_sample

sample_id_to_barcodes_passing_filters = {}
sample_id_to_thresholds = {}
for sample_id in fragments_dict:
    (
        sample_id_to_barcodes_passing_filters[sample_id],
        sample_id_to_thresholds[sample_id]
    ) = get_barcodes_passing_qc_for_sample(
            sample_id = sample_id,
            pycistopic_qc_output_dir = QC_DIR,
            unique_fragments_threshold = None, # use automatic thresholding
            tss_enrichment_threshold = None, # use automatic thresholding
            frip_threshold = 0,
            use_automatic_thresholds = True,
    )
print(sample_id_to_barcodes_passing_filters)

In [ ]:
print(sample_id_to_thresholds)

In [ ]:
# check how many cells passed these filters
for sample, barcodes in sample_id_to_barcodes_passing_filters.items():
    num_barcodes = len(barcodes)
    print(f"Sample: {sample}, Number of barcodes: {num_barcodes}")


In [ ]:
# Function to generate, display, and save the plot
def generate_and_save_barcode_plots(fragments_dict, QC_DIR):
    for sample_id in fragments_dict:
        fig = plot_barcode_stats(
            sample_id=sample_id,
            pycistopic_qc_output_dir=QC_DIR,
            bc_passing_filters=sample_id_to_barcodes_passing_filters[sample_id],
            detailed_title=False,
            **sample_id_to_thresholds[sample_id]
        )
        fig_path = os.path.join(QC_DIR, f'{sample_id}_barcode_qc_plot.png')
        fig.savefig(fig_path)  # Save the plot as a file
        print(f'Plot saved to {fig_path}')
        plt.show()  # Display the plot

generate_and_save_barcode_plots(fragments_dict, QC_DIR)

## Select ATAC barcodes to keep

In [ ]:
# Create a dictionary of the cells I want to keep based on the multiome integration 
cell_data =  pd.read_csv(projDir+'scenicplus_without_C18/input/forSCENIC_Epi_without_C18_Diet_seurat-wnn-integrated.tsv', sep='\t')


In [ ]:
# Iterate through each unique sample to create a dictionary
sample_data_dict = {}

for sample in cell_data['Sample'].unique():
    barcodes = cell_data.loc[cell_data['Sample'] == sample, 'barcode'].tolist()     # Filter the DataFrame for the current sample and retrieve barcodes
    sample_data_dict[sample] = barcodes     # Store the list of barcodes in the dictionary

#print(sample_data_dict)
print(len(sample_data_dict))  # Number of samples

for sample, barcodes in sample_data_dict.items():
    print(f"Sample '{sample}' has {len(barcodes)} barcodes")

## Create a Cistopic object from everything computed before and multiome integrated barcodes

In [ ]:
path_to_regions = os.path.join(outDir, "consensus_peak_calling/consensus_regions.bed")
pycistopic_qc_output_dir = QC_DIR

In [ ]:
from pycisTopic.cistopic_class import create_cistopic_object_from_fragments
import polars as pl

In [ ]:
fragments_dict

In [ ]:
# create a cistopic obj list with my samples, using multiome integrated barcodes 
# https://github.com/aertslab/pycisTopic/issues/131
cistopic_obj_list = []

In [ ]:
# loop killed the kernel twice, so I will do this individually
# for sample_id in fragments_dict:

sample_id = "Pilot1"

cistopic_obj = create_cistopic_object_from_fragments(
        path_to_fragments = fragments_dict[sample_id],
        path_to_regions = path_to_regions,
        path_to_blacklist = path_to_blacklist,
        valid_bc = sample_data_dict[sample_id],
        project = sample_id
    )
cistopic_obj_list.append(cistopic_obj)

In [ ]:
sample_id = "Batch1"

cistopic_obj = create_cistopic_object_from_fragments(
        path_to_fragments = fragments_dict[sample_id],
        path_to_regions = path_to_regions,
        path_to_blacklist = path_to_blacklist,
        valid_bc = sample_data_dict[sample_id],
        project = sample_id
    )
cistopic_obj_list.append(cistopic_obj)

In [ ]:
cistopic_obj_list

In [ ]:
outDir

In [ ]:
import pickle
pickle.dump(
    cistopic_obj_list, # not merged
    open(os.path.join(outDir, "cistopic_obj_list.pkl"), "wb")
)

In [ ]:
# merge list 
cistopic_mergedobj = cistopic_obj_list[0]
cistopic_mergedobj.merge(cistopic_obj_list[1:])
print(cistopic_mergedobj)

# **Match RNA Adata to the ATAC cistopic object to run Models**

In [ ]:
# check barcodes in the anndata 
adata = sc.read_h5ad(work_dir+'/adata_diet.h5ad')
adata.obs_names
adata.obs

In [ ]:
# check names in the cistopic object 
cistopic_mergedobj.cell_names

In [ ]:
len(cistopic_mergedobj.cell_names)

In [ ]:
# inspect differences in cell number 
cell_in_cistopic_not_anndata = set(cistopic_mergedobj.cell_names) - set(adata.obs["barcode"])
len(cell_in_cistopic_not_anndata)

In [ ]:
len(set(cistopic_mergedobj.cell_names) - set(adata.obs_names))

In [ ]:
cell_in_anndata_not_cistopic = set(adata.obs["barcode"]) - set(cistopic_mergedobj.cell_names)
len(cell_in_anndata_not_cistopic)

In [ ]:
# match adata file to the cistopic one 
  # rearrange order of barcodes and sample info
new_obs_names = [
    f"{'_'.join(name.split('_')[1:])}___{name.split('_')[0]}"
    for name in adata.obs_names
]
new_obs_names

In [ ]:
# rename obs_names in the adata file
adata.obs_names = new_obs_names
print(adata.obs_names[:10]) # Verify the change

common_cells = set(cistopic_mergedobj.cell_names).intersection(set(adata.obs_names))
print(f"Number of common cells: {len(common_cells)}")

In [ ]:
work_dir

In [ ]:
# save
adata.write(work_dir+"/adata_diet_matchingids.h5ad")

In [ ]:
# since I created a new cistopic file with merged information, add metadata to it again:
# Adding cell information to cistopic obj
cell_data
# correct barcode nomenclature 
cell_data.index = cell_data['barcode'] + '___' + cell_data['Sample']
cell_data.index

cistopic_mergedobj.add_cell_data(cell_data)
print(cistopic_mergedobj.cell_data)

In [ ]:
outDir

In [ ]:
# save
import pickle
pickle.dump(
    cistopic_mergedobj, # merged
    open(os.path.join(outDir, "cistopic_mergedobj.pkl"), "wb")
)

In [ ]:
# access saved file if needed:
# with open(os.path.join(outDir, "cistopic_mergedobj.pkl"), "rb") as file: cistopic_mergedobj = pickle.load(file)

# **RUN MODELS**

## Mallet was previously installed in /mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus/Mallet-202108/bin/mallet 

In [ ]:
#!wget https://github.com/mimno/Mallet/releases/download/v202108/Mallet-202108-bin.tar.gz

In [ ]:
#!tar -xf Mallet-202108-bin.tar.gz

## Create a script and run with nohup

In [ ]:
os.mkdir("/mnt/scratch/maithe_2025/SCENICplus_withoutC18_temp/mallet") 

## Create the script named: pycistopicmodelling.sh
```sh
#!/bin/bash -l

## Job needs 1 nodes and 12 cores per node.
#PBS -l nodes=1:ppn=12

## Batch job name.
#PBS -N integration_pycistopic

pycistopic topic_modeling mallet -i cistopic_mergedobj.pkl -o modelsmerged.pkl -t 30 35 40 45 50 60 70 80 90 100 110 -p 12 -m 100 -k True -T /mnt/scratch/maithe_2025/SCENICplus_withoutC18_temp/mallet -b /mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus/Mallet-202108/bin/mallet
```

## Evaluate Models

In [ ]:
outDir

In [ ]:
import pickle
with open(os.path.join(outDir, "modelsmerged.pkl"), 'rb') as infile:
    modelsmerged = pickle.load(infile)

Choosing which model to use is very empirical. See [this](https://github.com/aertslab/pycisTopic/discussions/107) where the author says 'In practice we select the best model manually based on where the curves start flattening (i.e. the model with the lowest number of topics, where most metrics are (semi) maximised)'

In [ ]:
# import matplotlib.pyplot as plt
from pycisTopic.lda_models import evaluate_models
import matplotlib.pyplot as plt

modelmerged = evaluate_models(
    modelsmerged,
    select_model = 80,
    return_model = True,
    metrics = ['Arun_2010', 'Cao_Juan_2009', 'Minmo_2011', 'loglikelihood'],
    plot_metrics = True
)

fig.set_size_inches(30, 10)  # Increase the figure size significantly
fig.tight_layout(pad=1.0)  # Add padding to prevent cutoff

# Save the plot to a file with higher resolution
plt.savefig('model_merged_evaluation.png', dpi=200, bbox_inches='tight')

## Add chosen model to object

In [ ]:
# access saved file if needed:
import pickle
with open(os.path.join(outDir, "cistopic_mergedobj.pkl"), "rb") as file:
  cistopic_mergedobj = pickle.load(file)

In [ ]:
len(cistopic_mergedobj.cell_names)
len(cistopic_mergedobj.region_names)

In [ ]:
# add chosen model to object 
cistopic_mergedobj.add_LDA_model(modelmerged)
cistopic_mergedobj.add_LDA_model

In [ ]:
# save
pickle.dump(
    cistopic_mergedobj,
    open(os.path.join(outDir, "cistopic_mergedobj_model.pkl"), "wb")
)

# **CLUSTERING and VISUALIZATION**  
Following this [tutorial](https://github.com/aertslab/pycisTopic/blob/old/notebooks/Cortex_pycisTopic.ipynb)  
For these steps, the cell-topic contriibutions of the model will be used. 
I won't perform clustering as I  will use the scRNA-seq labels, but take a look to the find_clusters function otherwise.   


In [ ]:
os.mkdir(outDir + '/visualization/') 
from pycisTopic.clust_vis import *

In [ ]:
# Load cisTopic object if needed
#import pickle
#infile = open(outDir + '/cistopic_mergedobj_model.pkl', 'rb')
#cistopic_mergedobj = pickle.load(infile)
#infile.close()

In [ ]:
print(dir(cistopic_mergedobj))

## Run UMAP

In [ ]:
# run  umap - initialize projections['cell'] if necessary
print("Projections before initialization:", cistopic_mergedobj.projections)

if 'cell' not in cistopic_mergedobj.projections:
    cistopic_mergedobj.projections['cell'] = {}

print("Projections after initialization:", cistopic_mergedobj.projections)
run_umap(cistopic_mergedobj,
                 target  = 'cell', scale=False)

In [ ]:
# save
pickle.dump(
    cistopic_mergedobj,
    open(os.path.join(outDir, "cistopic_mergedobj_model.pkl"), "wb")
)

## Plot

In [ ]:
# plot cells and samples 
plot_metadata(cistopic_mergedobj,
                 reduction_name='UMAP',
                 variables=['Cellannotation', 'Sample'], # Labels from RNA and new clusters
                 target='cell', num_columns=2,
                 text_size=10,
                 dot_size=5,
                 figsize=(10,5),
                 save= outDir + '/visualization/cistopicmergedobj_dimensionality_reduction_label_uncorrected.pdf')
        

In [ ]:
# see cell-topic distribution
plot_topic(cistopic_mergedobj,
            reduction_name = 'UMAP',
            target = 'cell',
            num_columns=5,
            save= outDir + '/visualization/dimensionality_reduction_topic_uncorrected.pdf')
            

In [ ]:
# plot heatmap 
cell_topic_heatmap(cistopic_mergedobj,
                     variables = ['Cellannotation'],
                     scale = False,
                     legend_loc_x = 1.0,
                     legend_loc_y = -1.2,
                     legend_dist_y = -1,
                     figsize=(10,10),
                     save = outDir + '/visualization/heatmap_topic_contr.pdf')

# **BINARIZE TOPICS**  

- Load cisTopic object to create sets of regions on which we will run SCENIC+
- To binarize the topics means that for each topic we will have regions, we will have probability in each region to belong to a certain topic  
- Then for each topic we want to select regions with the highest probability to belong to that topic
  
- This can be done automatically by this "Otsu" threshold method where for each topic, a number of regions get selected  
- Or we can just take the top 3 thousand regions of each topic (you can decide on the number of regions you want to keep) 

- Otsu and Yen’s methods work well for topic-region distributions  
- However for some downstream analyses (e.g. deep learning) it may be convenient to use ntop to have balanced region sets  



In [ ]:
os.mkdir(outDir+'/topic_binarization')
from pycisTopic.topic_binarization import *

In [ ]:
region_bin_topics = binarize_topics(
  cistopic_mergedobj, method='otsu', 
  plot=True, num_columns=5, save= outDir + '/topic_binarization/otsu.pdf')

In [ ]:
region_bin_topics_top_3k = binarize_topics(
    cistopic_mergedobj, method='ntop', ntop = 3_000,
    plot=True, num_columns=5, save = outDir + '/topic_binarization/ntop.pdf')

In [ ]:
binarized_cell_topic = binarize_topics(
    cistopic_mergedobj,
    target='cell',
    method='li',
    plot=True,
    num_columns=5, nbins=100)

In [ ]:
from pycisTopic.topic_qc import topic_annotation
topic_annot = topic_annotation(
    cistopic_mergedobj,
    annot_var='Cellannotation',
    binarized_cell_topic=binarized_cell_topic,
    general_topic_thr = 0.2
)

In [ ]:
topic_annot

In [ ]:
if topic_annot.is_general.any():
    print("There are general topics")
else:
    print("There are no general topics")

In [ ]:
# save as a .csv
topic_annot.to_csv(os.path.join(outDir, "topics_annotation.csv"))

In [ ]:
# now save this into BED files 
from scenicplus.utils import region_names_to_coordinates
  # otsu
os.makedirs(os.path.join(outDir, "region_sets", "topics_otsu"), exist_ok = True)
for topic in region_bin_topics:
  region_names_to_coordinates(
      region_bin_topics[topic].index
  ).sort_values(
      ["Chromosome", "Start", "End"]
  ).to_csv(
      os.path.join(outDir, "region_sets", "topics_otsu", f"{topic}.bed"),
      sep = "\t", header = False, index = False
  )

In [ ]:
# top 3k
os.makedirs(os.path.join(outDir, "region_sets", "topics_top_3k"), exist_ok = True)
for topic in region_bin_topics:
  region_names_to_coordinates(
      region_bin_topics_top_3k[topic].index
  ).sort_values(
      ["Chromosome", "Start", "End"]
  ).to_csv(
      os.path.join(outDir, "region_sets", "topics_top_3k", f"{topic}.bed"),
      sep = "\t", header = False, index = False
  )

# **DARs - Differentially Accessible Regions**  
For each cell type, we can compute regions that are more accessible in this cell type compared to other cells in the dataset

In [ ]:
os.mkdir(outDir + '/DARs')

In [ ]:
# raw accessibility counts
from pycisTopic.diff_features import (
  impute_accessibility,
  normalize_scores,
  find_highly_variable_features,
  find_diff_features
)
import numpy as np

## Impute accessibility

In [ ]:
#  impute the region accessibility exploting the cell-topic and topic-region probabilities
# To shrink very low probability values to 0, we use a scale factor (by default: 10^6).
imputed_acc_obj = impute_accessibility(
  cistopic_mergedobj,
  selected_cells = None,
  selected_regions = None,
  scale_factor = 10**6
)

pickle.dump(
    imputed_acc_obj,
    open(os.path.join(outDir, "DARs", "imputed_acc_obj.pkl"), "wb"))

## Normalize imputed data

In [ ]:
import pickle

infile = open(outDir + '/DARs/imputed_acc_obj.pkl', 'rb')
imputed_acc_obj = pickle.load(infile)
infile.close()

In [ ]:
from pycisTopic.diff_features import (
  normalize_scores,
  find_highly_variable_features,
  find_diff_features
)

In [ ]:
normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

In [ ]:
imputed_acc_obj

In [ ]:
del imputed_acc_obj

In [ ]:
pickle.dump(normalized_imputed_acc_obj, open(os.path.join(outDir, "normalized_imputed_acc_obj.pkl"), "wb"))

In [ ]:
import gc
gc.collect()

## Identify highly variable regions

In [ ]:
import numpy as np

In [ ]:
# identify highly variable regions
variable_regions = find_highly_variable_features(normalized_imputed_acc_obj,
                                           min_disp = 0.05,
                                           min_mean = 0.0125, 
                                           max_mean = 3,
                                           max_disp = np.inf,
                                           n_bins=20, 
                                           n_top_features=None,
                                           plot=True,
                                           save= outDir + '/DARs/HVR_plot.pdf')
del normalized_imputed_acc_obj
pickle.dump(variable_regions, open(os.path.join(outDir, "/DARs/variable_regions.pkl"), "wb"))

**Find variable features**  

```sh

import os
os.environ["OMP_NUM_THREADS"] = "2" 
os.environ["MKL_NUM_THREADS"] = "2" 
os.environ["OPENBLAS_NUM_THREADS"] = "2" 
os.environ["NUMEXPR_NUM_THREADS"] = "2"

import scenicplus
import pycisTopic 
import scanpy as sc
import pandas as pd
import pickle
import numpy as np
import joblib
import matplotlib.pyplot as plt

outDir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus/output"

from pycisTopic.diff_features import find_highly_variable_features

# Load the imputed accessibility object
with open(os.path.join(outDir, 'normalized_imputed_acc_obj2.pkl'), 'rb') as infile:
    normalized_imputed_acc_obj = pickle.load(infile)

variable_regions = find_highly_variable_features(normalized_imputed_acc_obj, min_disp = 0.05, min_mean = 0.0125, max_mean = 3, max_disp = np.inf, n_bins=20, n_top_features=None, plot=False)  

pickle.dump(variable_regions, open(os.path.join(outDir, "DARs", "variable_regions.pkl"), "wb"))

```

## Identify DAR between cell types and save as bed files

In [ ]:
import pickle
with open(os.path.join(outDir, 'cistopic_mergedobj_model.pkl'), 'rb') as infile:
    cistopic_mergedobj = pickle.load(infile)

In [ ]:
cistopic_mergedobj.cell_data['Cellannotation'].unique()

In [ ]:
infile = open(outDir + '/DARs/variable_regions.pkl', 'rb')
variable_regions = pickle.load(infile)
infile.close()

In [ ]:
variable_regions

In [ ]:
with open(os.path.join(outDir, 'DARs', 'imputed_acc_obj.pkl'), 'rb') as infile:
    imputed_acc_obj = pickle.load(infile)

In [ ]:
imputed_acc_obj

In [ ]:
from pycisTopic.cistopic_class import *
from pycisTopic.diff_features import *

In [ ]:
markers_dict= find_diff_features(cistopic_mergedobj, 
                      imputed_acc_obj,
                      variable='Cellannotation',
                      var_features=variable_regions,
                      contrasts=None,
                      adjpval_thr=0.05,
                      log2fc_thr=np.log2(1.5),
                      _temp_dir = tmpDir,
                      split_pattern = '-')

In [ ]:
# save
pickle.dump(
    markers_dict,
    open(os.path.join(outDir, "DARs", "markers_dict.pkl"), "wb"))

In [ ]:
# open if neccessary 
# with open(os.path.join(outDir, "DARs", "markers_dict.pkl"), 'rb') as infile:
#    markers_dict = pickle.load(infile)

In [ ]:
os.makedirs(os.path.join(outDir, "region_sets", "DARs_cell_type"), exist_ok = True)
from pycisTopic.utils import region_names_to_coordinates

for cell_type in markers_dict:
  region_names_to_coordinates(
      markers_dict[cell_type].index
  ).sort_values(
      ["Chromosome", "Start", "End"]
  ).to_csv(
      os.path.join(outDir, "region_sets", "DARs_cell_type", f"{cell_type}.bed"),
      sep = "\t", header = False, index = False
  )

# **CISTARGET DATABASE**  
Follow the steps [here](https://scenicplus.readthedocs.io/en/latest/human_cerebellum_ctx_db.html)  

1 - Download create_cistarget database  
2 - Download cluster-buster  
3 - Download motif-collection  
4 - Prepare fasta from consensus regions  
5 - Create cistarget databases  


**Download genome and chromosome annotation files to prepare database**
```sh
mkdir /mnt/data/maithe_2025/genomes_2025
cd /mnt/data/maithe_2025/genomes_2025
wget ftp://ftp.ensembl.org/pub/release-101/fasta/mus_musculus/dna/Mus_musculus.GRCm38.dna.primary_assembly.fa.gz
gunzip Mus_musculus.GRCm38.dna.primary_assembly.fa.gz
http://hgdownload.soe.ucsc.edu/goldenPath/mm10/bigZips/mm10.chrom.sizes
```

**HPC: download create_cistarget_database:**

```sh
cd /mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/
conda activate scenicplus_3-11-8
mkdir ctx_db
cd ctx_db

git clone https://github.com/aertslab/create_cisTarget_databases
```

**download cluster-buster**  

```sh
wget https://resources.aertslab.org/cistarget/programs/cbust
chmod a+x cbust
```

**download motif collection**

```sh
mkdir -p aertslab_motif_collection
wget -O aertslab_motif_collection/v10nr_clust_public.zip https://resources.aertslab.org/cistarget/motif_collections/v10nr_clust_public/v10nr_clust_public.zip
cd aertslab_motif_collection; unzip -q v10nr_clust_public.zip
cd ..

ls aertslab_motif_collection/v10nr_clust_public/snapshots/
ls -l aertslab_motif_collection/v10nr_clust_public/singletons | head

cat aertslab_motif_collection/v10nr_clust_public/singletons/bergman__Adf1.cb

```

**converting fasta file header to ucsc**


```sh 
sed -E 's/^>([0-9]+|X|Y|MT)[^ ]*/>chr\1/' Mus_musculus.GRCm38.dna.primary_assembly.fa > Mus_musculus.GRCm38.dna.primary_assembly.ucsc.fa
```

**scritp named prepate_fasta_from_consensus_regions_GRCm38.sh**
```sh
#!/bin/bash

export OMP_NUM_THREADS=16

REGION_BED="/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus/output/consensus_peak_calling/consensus_regions.bed"
GENOME_FASTA="/mnt/data/maithe_2025/genomes_2025/Mus_musculus.GRCm38.dna.primary_assembly.ucsc.fa"
CHROMSIZES="/mnt/data/maithe_2025/genomes_2025/mm10.chrom.sizes"
DATABASE_PREFIX="healthy_reannotated_1kb_bg_with_mask"
SCRIPT_DIR="/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus/ctx_db/create_cisTarget_databases"

${SCRIPT_DIR}/create_fasta_with_padded_bg_from_bed.sh \
    ${GENOME_FASTA} \
    ${CHROMSIZES} \
    ${REGION_BED} \
    GRCm38_ucsc_with_1kb_padding.fa \
    1000 \
    yes
```

**Create script to Create cistarget databases**

``` sh
cd /mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/ctx_db
ls aertslab_motif_collection/v10nr_clust_public/singletons > motifs.txt

```


```sh
#!/bin/bash

export OMP_NUM_THREADS=16
export PATH="$PWD:$PATH"

OUT_DIR=""${PWD}""
CBDIR="${OUT_DIR}/aertslab_motif_collection/v10nr_clust_public/singletons"
FASTA_FILE="${OUT_DIR}/GRCm38_ucsc_with_1kb_padding.fa"
MOTIF_LIST="${OUT_DIR}/motifs.txt"
DATABASE_PREFIX="healthy_reannotated_1kb_bg_with_mask" 
SCRIPT_DIR="/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/ctx_db/create_cisTarget_databases"  

"${SCRIPT_DIR}/create_cistarget_motif_databases.py" \
    -f ${FASTA_FILE} \
    -M ${CBDIR} \
    -m ${MOTIF_LIST} \
    -o ${OUT_DIR}/${DATABASE_PREFIX} \
    --bgpadding 1000 \
    -t 20
```

**Download motif annotation into /input/**

```sh
wget "https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl"
```